# Step 6 — Neural pattern-shift vs within-person sentiment change
**The most direct test of the locked question:** on runs where a person's expressed sentiment about a
character *shifted* a lot, did their *neural pattern* shift a lot too (around that character's "aha"
moments)? This is Jin's `step08` (`neural_pattern_shift ~ impression_update`) with **our sentiment deltas
swapped in for his USE impression-updates** — the behavioral side built in Step 0 (`results/deltas/`).

**Status: run.** Jin provided the BOLD data (via Box) and the aha annotations, so both sides now run
end-to-end. Because `scene_null.py` is absent from his public repo, the permutation null is reconstructed
faithfully from the non-aha sampler documented in Jin's `step08` and the paper Methods p.23 (see **6.2-corrected**). The port was verified line-by-line against `step08` source.

> [!important] Result — clean null
> The faithful `step08` port (per-TR FDR over the 100 cortical ROIs, two-tailed, and the step07∩step08
> double threshold) returns **0 significant ROIs at every TR (−5 … +3)**. So within-person sentiment
> *change* does not track neural pattern *shift* around aha moments at this threshold. Reported as-is.

### Brain inputs this uses (beyond what IS-RSA / `04b` needs)
1. **`data/brain/pattern_shift/1TR_nearbytp.npy`** — TR-by-TR neural pattern-shift per (group, ROI, subject)
   (Jin's `step06` output).
2. **`data/beh/annotations/ahaannot_all.xlsx`** — 3-rater "aha" annotations (character-aha timepoints:
   `character_all ≥ 2`, with per-run/scene TR positions).
3. the group-scene event CSV, and — for surface figures — nilearn + Schaefer/Tian templates.

> [!note] Hemodynamic lag (hrf) — tested both ways; the null is robust
> The pattern shift is built with **`hrf=3`** (Jin's `step06`); his email, the paper, `step01`, and
> `code_extractbold.py` say **4**. Ran both (6.3): the manuscript-faithful **double threshold is 0 ROIs
> under hrf=3 AND hrf=4**, so the sentiment pattern-shift null does not depend on the lag. (Under hrf=4,
> step08-only surfaces a lone cortical ROI [19] at TR−4, but it's not in Jin's step07 set for that TR, so it
> fails the double threshold.) Still worth confirming which hrf the published Fig 4 used — for correctness,
> not the conclusion. See `CODE_VS_PAPER_MISMATCHES.md` #1.

## 6.0 · Paths + ported helpers

In [3]:
import os, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH

# Anchor DELTA_DIR specifically to the absolute JIN_REPO path
DELTA_DIR   = Path("results/deltas")  # our Step-0 sentiment deltas (behavioral side) -- present
MODEL       = "Twitter_RoB"
from config import JIN_REPO
PATTERN_SHIFT_PATH = os.path.join(JIN_REPO, "data/brain/pattern_shift/1TR_nearbytp.npy")
from config import AHA_ANNOT_PATH
from config import EVENT_PATH
# --------------------------
NROI = 116
HAVE_DELTAS = (DELTA_DIR / f"00__delta_profiles_{MODEL}.csv").exists()
HAVE_SHIFT  = os.path.exists(PATTERN_SHIFT_PATH)
HAVE_AHA    = os.path.exists(AHA_ANNOT_PATH)
HAVE_EVENT  = os.path.exists(EVENT_PATH)
print("deltas:",HAVE_DELTAS,"| pattern_shift:",HAVE_SHIFT,"| aha annot:",HAVE_AHA,"| event csv:",HAVE_EVENT)

_JIN_FLIST={1:["sub-1001","sub-1005","sub-1008","sub-1011","sub-1014","sub-1017","sub-1020","sub-1023","sub-1026","sub-1029","sub-1033","sub-1039"],
            2:["sub-2006","sub-2009","sub-2012","sub-2015","sub-2018","sub-2021","sub-2024","sub-2027","sub-2034","sub-2038","sub-2040"],
            3:["sub-3004","sub-3007","sub-3013","sub-3016","sub-3019","sub-3022","sub-3025","sub-3031","sub-3037","sub-3041"]}
def conv_r2z(r):
    with np.errstate(invalid="ignore", divide="ignore"): return 0.5*(np.log(1+r)-np.log(1-r))
def nanspearmanr(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    idx=np.union1d(np.where(np.isnan(a))[0],np.where(np.isnan(b))[0])
    return spearmanr(np.delete(a,idx),np.delete(b,idx))[0]

deltas: True | pattern_shift: True | aha annot: True | event csv: True


In [2]:
# 6.0c · Generate the neural pattern-shift from Jin's loaded_BOLD pkls
import os, pickle, numpy as np, pandas as pd
import warnings, gc
from config import LOADED_BOLD_DIR

EVENTS_DIR      = os.path.join(JIN_REPO, "data/brain/events")          
OUT_DIR         = os.path.join(JIN_REPO, "data/brain/pattern_shift"); os.makedirs(OUT_DIR, exist_ok=True)
flist=_JIN_FLIST; tasklist=[f"{i:02d}" for i in range(1,11)]; nroi=116; window=1
hrf=3   # matches Jin's step06 (his published pattern shift). NB: step01/code_extractbold/paper/email say 4 -- test hrf=4 too. See CODE_VS_PAPER_MISMATCHES.md #1.

if all(os.path.exists(os.path.join(LOADED_BOLD_DIR,f"ROIsum_combined_mask_g{g}.pkl")) for g in [1,2,3]):
    diss = {}
    for g in [1,2,3]:
        pkl_path = os.path.join(LOADED_BOLD_DIR, f"ROIsum_combined_mask_g{g}.pkl")
        print(f"Loading Group {g} into memory (this may take a minute)...")
        
        with open(pkl_path, "rb") as f: 
            bold = pickle.load(f)
        
        # 1. Hoist I/O: Pre-calculate TRs
        sub_run_TRs = {}
        for sub in range(len(flist[g])):
            sub_run_TRs[sub] = {}
            for run in range(1, 10):
                if run == 6: continue
                ev_path = os.path.join(EVENTS_DIR, f"{flist[g][sub]}_task-{tasklist[run]}_events.tsv")
                ev = pd.read_csv(ev_path, sep="\t")
                sub_run_TRs[sub][run] = int(ev["onset"][4] + ev["duration"][4])
        
        print(f"Group {g} loaded. Processing ROIs and Subjects...")
        
        for roi in range(1, nroi+1):
            for sub in range(len(flist[g])):
                bysub = []
                for run in range(1, 10):
                    if run == 6: continue
                    
                    TRs = sub_run_TRs[sub][run]
                    tc = bold[sub][run][roi][:, hrf:TRs+hrf]
                    
                    # Filter out invalid voxels
                    valid_mask = np.all(np.isfinite(tc), axis=1)
                    tc_clean = tc[valid_mask, :]
                    
                    if tc_clean.shape[0] < 2:
                        bysub.append(np.full(tc.shape[1] - 2 * window, np.nan))
                    else:
                        # 2. Vectorize Math
                        X = tc_clean[:, :tc_clean.shape[1] - 2 * window]
                        Y = tc_clean[:, window : tc_clean.shape[1] - window]
                        
                        X_c = X - np.mean(X, axis=0)
                        Y_c = Y - np.mean(Y, axis=0)
                        
                        num = np.sum(X_c * Y_c, axis=0)
                        den = np.sqrt(np.sum(X_c**2, axis=0) * np.sum(Y_c**2, axis=0))
                        
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            corr = num / den
                            
                        shifts = 1 - corr
                        bysub.append(shifts)
                        
                diss[g,roi,sub] = bysub
        
        # 3. CRITICAL MEMORY RELEASE
        # Force Python to delete the massive BOLD dictionary and clear RAM 
        # before attempting to load the next group's .pkl file.
        print(f"Group {g} complete. Forcing memory garbage collection...")
        del bold
        gc.collect() 
                
    np.save(os.path.join(OUT_DIR,"1TR_nearbytp.npy"), diss, allow_pickle=True)
    print("saved", os.path.join(OUT_DIR,"1TR_nearbytp.npy"))
else:
    print("Place Jin's ROIsum_combined_mask_g{1,2,3}.pkl in", LOADED_BOLD_DIR, "then re-run.")

Loading Group 1 into memory (this may take a minute)...
Group 1 loaded. Processing ROIs and Subjects...
Group 1 complete. Forcing memory garbage collection...
Loading Group 2 into memory (this may take a minute)...
Group 2 loaded. Processing ROIs and Subjects...
Group 2 complete. Forcing memory garbage collection...
Loading Group 3 into memory (this may take a minute)...
Group 3 loaded. Processing ROIs and Subjects...
Group 3 complete. Forcing memory garbage collection...
saved /Users/rheamadhogarhia/Desktop/CABLAB RESEARCH/independent ra work/part_4_summer_char_profiles/characterprofilesynchronygit/data/socialaha/data/brain/pattern_shift/1TR_nearbytp.npy


In [2]:
# 6.0b Preflight -- which inputs "resolve" (= exist on disk) on this machine?
def check_pattern_shift_inputs():
    items = {
        "sentiment deltas (behavioral side)":                 str(DELTA_DIR/f"00__delta_profiles_{MODEL}.csv"),
        "neural pattern-shift 1TR_nearbytp.npy (Jin step06)": PATTERN_SHIFT_PATH,
        "   fallback: loaded BOLD dir (Jin step01)":          os.path.join(JIN_REPO, "data/brain/loaded_BOLD"),
        "aha annotations ahaannot_all.xlsx":                  AHA_ANNOT_PATH,
        "group-scene event CSV":                              EVENT_PATH,
    }
    print("pattern-shift input check (does each path resolve = exist on disk?):")
    missing = []
    for label, p in items.items():
        ok = os.path.exists(p)
        print(f"  [{'OK     ' if ok else 'MISSING'}] {label}")
        if (not ok) and (not label.strip().startswith("fallback")): missing.append(label.strip())
    print("\nAll required inputs resolve -- 06 can run." if not missing
          else f"\n{len(missing)} required input(s) do NOT resolve: {missing}")
    return not missing
check_pattern_shift_inputs()

pattern-shift input check (does each path resolve = exist on disk?):
  [OK     ] sentiment deltas (behavioral side)
  [OK     ] neural pattern-shift 1TR_nearbytp.npy (Jin step06)
  [MISSING]    fallback: loaded BOLD dir (Jin step01)
  [OK     ] aha annotations ahaannot_all.xlsx
  [OK     ] group-scene event CSV

All required inputs resolve -- 06 can run.


True

## 6.1 · Behavioral side — within-person sentiment change (swaps in for USE impression-update)

For each subject × character × consecutive-run pair, the **|Δ valence|** = |Δpos − Δneg| from our Step-0
delta profiles. This is the sentiment analog of Jin's `1 − cosine(embedding_run, embedding_run−1)`. Runs
now (deltas are present).

In [3]:
if HAVE_DELTAS:
    dd = pd.read_csv(DELTA_DIR/f"00__delta_profiles_{MODEL}.csv")
    dd["Character"] = dd["Character"].str.lower().str.strip()
    dd["sent_update"] = (dd[f"d_{MODEL}_pos"] - dd[f"d_{MODEL}_neg"]).abs()   # |Δ valence| per consecutive-run
    dd["group"] = dd["Participant"].astype(str).str.extract(r"(\d)").astype(int)
    sent_update = dd[["Participant","group","Character","Run_from","Run_to","sent_update"]]
    print("sentiment-update rows:", len(sent_update), "| participants:", sent_update.Participant.nunique())
    print(sent_update.head(4).to_string(index=False))
    print("\nNext step (needs the brain inputs): align these to Jin's 40 (subject x scene) grid via the")
    print("event CSV, exactly as step08's compute_impression_updates does, then correlate vs neural shift.")
else:
    sent_update=None; print("Set DELTA_DIR / run Step 0 first.")

sentiment-update rows: 1144 | participants: 32
Participant  group Character  Run_from  Run_to  sent_update
   sub-1001      1      jack         1       2     0.839687
   sub-1001      1      jack         2       3     0.567508
   sub-1001      1      jack         3       4     1.624659
   sub-1001      1      jack         4       5     1.253461

Next step (needs the brain inputs): align these to Jin's 40 (subject x scene) grid via the
event CSV, exactly as step08's compute_impression_updates does, then correlate vs neural shift.


In [5]:
import pandas as pd
from config import AHA_ANNOT_PATH

df = pd.read_excel(AHA_ANNOT_PATH)
print(df.columns.tolist())

['subject', 'run', 'scene', 'TR (scene)', 'TR (run)', 'retrieval', 'retrieved scenes', 'current', 'character', 'relationship', 'inference', 'temporal', 'oops', 'causal', 'description']


## 6.2 · Pattern-shift ~ sentiment-update — faithful Jin `step08` port (corrected)

This replaces the earlier on-the-fly-null draft. Three corrections, all traced to Jin's **public** `step08` source (`github.com/jinke828/socialaha`):

1. **Null.** `scene_null.py` and its output `null_nonearaha_1TR_nb_perscene.npy` are **not in Jin's public repo** (verified — the file 404s and is absent from the local clone). The only documented sampler is `step07`'s: draw random **non-aha** timepoints and take their `[-5,+3]` windows. This cell reconstructs that at the per-scene level — each scene draws a number of non-aha windows **equal to its real aha count** and averages them, so the null is structurally identical to the actual statistic and only the aha *positions* are randomized.
2. **Estimator.** Actual and null use one tie-aware, NaN-exact Spearman that equals `scipy.spearmanr` to 1e-17 (replaces the `argsort`-`argsort` ranks, which broke ties differently from SciPy).
3. **Significance.** Jin's exact method: per-TR two-tailed permutation *p*, FDR over the **100 cortical** ROIs, then the **double threshold** (intersect with `step07`'s elevated-shift ROIs). The earlier joint-116×9 FDR was not his method and is what inflated the counts.

⚠️ **Runtime:** the faithful null is cluster-scale (that is *why* Jin precomputed it). At `N_PERM=1000` this runs for a long time on a laptop — lower `N_PERM` for a quick look (*p*-resolution = 1/(N+1)). See the provenance cell below for what is inferred vs. certain.


In [ ]:
# 6.2 · Pattern-shift ~ sentiment-update -- FAITHFUL port that CALLS JIN'S OWN step08 code.
# Jin's exact functions are imported from jin_code/jin_step08.py (extracted verbatim from his repo, attributed).
# Only TWO things deviate from his step08, both marked ★:
#   ★ behavioral input = our sentiment |Δvalence|  (replaces his compute_impression_updates / USE embeddings)
#   ★ permutation null  = RECONSTRUCTED per-scene non-aha draw (his scene_null.py output is absent from the repo)
# Everything else -- compute_shifts_perscene, compute_rvals, test_significance, double_threshold -- is JIN'S exact
# code, called unchanged. Verified: J.compute_rvals reproduces this cell's earlier actual_rvals to max|diff|=0.
import numpy as np, pandas as pd, sys, os
sys.path.insert(0, "jin_code"); import jin_step08 as J
from scipy.stats import spearmanr, rankdata
N_PERM, SEED = 1000, 42
if HAVE_DELTAS and HAVE_SHIFT and HAVE_AHA and HAVE_EVENT:
    event_idxs    = pd.read_csv(EVENT_PATH)
    pattern_shift = np.load(PATTERN_SHIFT_PATH, allow_pickle=True).item()
    df_full = pd.read_excel(AHA_ANNOT_PATH)
    if 'character' in df_full.columns: df_char = df_full[df_full['character']==1].copy()
    else:
        df_full['character_all']=df_full[['character_rater1','character_rater2','character_rater3']].sum(1); df_char=df_full[df_full['character_all']>=2].copy()
    # ── ★ BEHAVIORAL: our sentiment update (the ONLY swap from Jin's compute_impression_updates) ──
    CHAR=["jack","kate","randall","kevin"]; _cid=lambda g,r: sorted(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())
    all_subs=[]
    for g in range(1,4):
        for ss in _JIN_FLIST[g]:
            row=[]
            for run in range(2,11):
                if run==7: continue
                dd=[]
                for ch in CHAR:
                    m=(sent_update['Participant']==ss)&(sent_update['Run_to']==run)&(sent_update['Character']==ch)
                    v=sent_update[m]['sent_update'].values; dd.append(v[0] if len(v) else np.nan)
                for idx in _cid(g,run): row.append(dd[int(idx)-1] if idx<5 else (dd[1]+dd[3])/2)  # = Jin id=5 kate+kevin
            all_subs.append(np.array(row))
    all_subs=np.array(all_subs)
    # ── NEURAL SIDE + r-values: JIN'S EXACT functions ──
    df_lookup       = J.build_df_lookup(df_char, _JIN_FLIST)                                   # JIN
    shifts_perscene = J.compute_shifts_perscene(df_lookup, pattern_shift, _JIN_FLIST, 116, event_idxs)  # JIN
    rvals           = J.compute_rvals(shifts_perscene, all_subs)                               # JIN
    # ── ★ RECONSTRUCTED per-scene non-aha null (scene_null.py absent) -> then Jin's r-value math ──
    _r2z=J.conv_r2z
    def _spb(Z,y):
        out=np.full(Z.shape[0],np.nan); vy=~np.isnan(y)
        if vy.sum()<2: return out
        Zc,yc=Z[:,vy],y[vy]; cl=~np.isnan(Zc).any(1)
        if cl.any():
            yr=rankdata(yc)-np.mean(rankdata(yc)); Zr=rankdata(Zc[cl],axis=1); Zr=Zr-Zr.mean(1,keepdims=True)
            den=np.linalg.norm(Zr,axis=1)*np.linalg.norm(yr)
            with np.errstate(all="ignore"): out[np.where(cl)[0]]=np.where(den>1e-8,Zr@yr/den,np.nan)
        for i in np.where(~cl)[0]:
            m=~np.isnan(Zc[i]); out[i]=spearmanr(Zc[i][m],yc[m])[0] if m.sum()>2 else np.nan
        return out
    nl={}
    for g in range(1,4):
        for si,ss in enumerate(_JIN_FLIST[g]):
            ds=df_char[df_char['subject']==int(ss[4:])]
            for run in range(8):
                ri=run+2 if run<5 else run+3; nl[(g,si,run)]=ds[ds['run']==ri]['TR (run)'].tolist()
    _as=lambda g,r: np.argsort(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())
    rng=np.random.default_rng(SEED); null_rvals=np.zeros((N_PERM,116,9))
    for roi in range(116):
        subs=[]
        for g in range(1,4):
            for si in range(len(_JIN_FLIST[g])):
                runs=[]
                for run in range(8):
                    ri=run+2 if run<5 else run+3; this=pattern_shift[g,roi+1,si][run]; L=len(this); ex=set()
                    for t in nl[(g,si,run)]: ex.update(range(max(0,t-5),min(L,t+4)))
                    av=np.array([t for t in range(5,L-3) if t not in ex]); scenes=np.full((N_PERM,5,9),np.nan)
                    for s in range(1,6):
                        k=len(df_lookup[(g,si,run,s)])
                        if k>0 and len(av)>0:
                            ch=rng.choice(av,size=(N_PERM,k)); scenes[:,s-1,:]=np.nanmean(this[ch[:,:,None]+np.arange(-5,4)[None,None,:]],1)
                    runs.append(scenes[:,_as(g,ri),:])
                subs.append(np.concatenate(runs,axis=1))
        nb_=np.stack(subs,1); zc=_r2z(nb_)
        for tr in range(9): null_rvals[:,roi,tr]=np.nanmean(np.stack([_spb(zc[:,:,sc,tr],all_subs[:,sc]) for sc in range(40)],1),1)
        if roi%20==0: print("null roi",roi)
    # ── SIGNIFICANCE: JIN'S EXACT test_significance + double_threshold ──
    sig_step08 = J.test_significance(rvals, null_rvals, nroi_cor=100)                          # JIN
    sig_step07 = [   # VERBATIM from Jin's step08 main() -- his hardcoded elevated-shift (step07) ROIs
        np.array([],dtype=int), np.array([81,88]), np.array([],dtype=int),
        np.array([6,15,20,25,27,29,30,31,32,33,36,38,40,41,42,43,44,46,48,49,66,74,75,76,79,80,81,83,84,85,86,87,88,89,92,93,94,98]),
        np.array([2,6,13,15,20,25,27,29,30,31,33,35,36,39,42,43,44,46,49,65,66,74,75,79,80,81,82,83,84,85,87,88,91,92,98,99]),
        np.array([2,6,13,15,19,20,23,25,28,29,30,31,32,33,35,36,38,40,42,43,44,49,50,51,65,66,74,75,79,80,81,82,83,84,85,87,88,92,94,95,97,98,99]),
        np.array([2,6,13,14,15,25,30,32,35,44,49,65,66,74,79,80,83,84,85,87]), np.array([],dtype=int), np.array([],dtype=int)]
    sig_double = J.double_threshold(rvals, null_rvals, sig_step07, nroi_cor=100)               # JIN
    print("\nSTEP08 only (Jin test_significance):")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_step08[tr]):2d}  {[int(x) for x in sig_step08[tr]]}")
    print("\nDOUBLE THRESHOLD (Jin double_threshold, hrf=3):")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_double[tr]):2d}  {[int(x) for x in sig_double[tr]]}")
    os.makedirs("results/step6",exist_ok=True)
    np.save("results/step6/06__pattern_shift_sentiment_hrf3.npy",
            {"actual_rvals":rvals,"HRF":3,"sig_step08":np.array(sig_step08,dtype=object),"sig_double":np.array(sig_double,dtype=object)},allow_pickle=True)
    print("\nSaved -> results/step6/06__pattern_shift_sentiment_hrf3.npy")
else:
    print("Cannot run: place pattern_shift, aha annot, event csv; run 6.1 (sent_update) first.")

STEP08 only (Jin test_significance):
  TR -5:  0  []
  TR -4:  0  []
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

DOUBLE THRESHOLD (Jin double_threshold, hrf=3):
  TR -5:  0  []
  TR -4:  0  []
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

Saved -> results/step6/06__pattern_shift_sentiment_hrf3.npy


> [!note] Runtime
> The faithful null is cluster-scale (that is *why* Jin precomputed it). This `N_PERM=1000` run took
> **~276 min** on a laptop; lower `N_PERM` for a quick look (*p*-resolution = 1/(N+1)).

> [!note] Methods note (for the write-up / email to Jin)
> Because `scene_null.py` (and its output `null_nonearaha_1TR_nb_perscene.npy`) is not in Jin's public
> repo, the permutation null was reconstructed faithfully from the non-aha sampling logic documented in
> `step07`: each scene draws a number of non-aha `[−5, +3]` windows equal to its real aha count, so the
> null is structurally identical to the statistic and only the aha *positions* are randomized. The *correlation* step matches Jin's `step08` (`compute_and_save_scene_null`), and the non-aha structure
> matches the paper Methods (p.23) — but `step08` only **loads** the pre-shuffled null (`scene_null.py`'s missing
> output); the **draw itself (per-scene count + averaging) is inferred** from `step07`'s sampler, so this is a
> reconstruction pending Jin's confirmation, not a bit-for-bit port. Under this faithful null, the
> pattern-shift ~ sentiment-update test yields **0 significant ROIs** (per-TR FDR over
> the 100 cortical ROIs, and the step07∩step08 double threshold).

## 6.3 · Sensitivity — hrf = 4 (Jin's stated value) vs the hrf = 3 run above

Jin's `step06` (his published pattern shift) uses **hrf = 3**, which 6.2 replicates. But his email, the paper,
`step01`, and `code_extractbold.py` all say **hrf = 4**. Since 3 vs 4 slides every `[-5,+3]` window one TR against
the aha stamps, this reruns the whole thing at hrf = 4 and **saves to separate files** so both are kept:

- neural input: `data/brain/pattern_shift/1TR_nearbytp_hrf4.npy` (6.3a) — the hrf=3 file is untouched
- result: `results/step6/06__pattern_shift_sentiment_hrf4.npy` (6.3b) vs `..._hrf3.npy` from 6.2

Run 6.3a then 6.3b **locally** (the regen loads the ~25 GB BOLD pkls; the null is the ~276-min step). Compare the
double-threshold rows: if hrf=4 also gives 0 ROIs, the null is robust to the lag; if it lights up, the lag choice
mattered and we take it to Jin. See `CODE_VS_PAPER_MISMATCHES.md` #1.

**Result (ran 2026-07-16):** double threshold = **0 ROIs at every TR under both hrf=3 and hrf=4** — the null is
robust to the lag. Under hrf=4, `step08`-only shows a single ROI [19] at TR−4 that does **not** survive the double
threshold (not in `step07`'s TR−4 set [81, 88]).

In [4]:
# 6.3a · SENSITIVITY -- regenerate the pattern shift at hrf=4 into a SEPARATE file (keeps hrf=3 intact)
# 6.0c · Generate the neural pattern-shift from Jin's loaded_BOLD pkls
import os, pickle, numpy as np, pandas as pd
import warnings, gc
from config import LOADED_BOLD_DIR

EVENTS_DIR      = os.path.join(JIN_REPO, "data/brain/events")          
OUT_DIR         = os.path.join(JIN_REPO, "data/brain/pattern_shift"); os.makedirs(OUT_DIR, exist_ok=True)
flist=_JIN_FLIST; tasklist=[f"{i:02d}" for i in range(1,11)]; nroi=116; window=1
hrf=4   # SENSITIVITY twin of the 6.0c cell -- Jin's stated value (email/paper/step01/code_extractbold)

HRF4_SHIFT = os.path.join(OUT_DIR,"1TR_nearbytp_hrf4.npy")
if os.path.exists(HRF4_SHIFT):
    print("hrf=4 pattern shift already exists at", HRF4_SHIFT, "-- skipping regen (delete it to force).")
elif all(os.path.exists(os.path.join(LOADED_BOLD_DIR,f"ROIsum_combined_mask_g{g}.pkl")) for g in [1,2,3]):
    diss = {}
    for g in [1,2,3]:
        pkl_path = os.path.join(LOADED_BOLD_DIR, f"ROIsum_combined_mask_g{g}.pkl")
        print(f"Loading Group {g} into memory (this may take a minute)...")
        
        with open(pkl_path, "rb") as f: 
            bold = pickle.load(f)
        
        # 1. Hoist I/O: Pre-calculate TRs
        sub_run_TRs = {}
        for sub in range(len(flist[g])):
            sub_run_TRs[sub] = {}
            for run in range(1, 10):
                if run == 6: continue
                ev_path = os.path.join(EVENTS_DIR, f"{flist[g][sub]}_task-{tasklist[run]}_events.tsv")
                ev = pd.read_csv(ev_path, sep="\t")
                sub_run_TRs[sub][run] = int(ev["onset"][4] + ev["duration"][4])
        
        print(f"Group {g} loaded. Processing ROIs and Subjects...")
        
        for roi in range(1, nroi+1):
            for sub in range(len(flist[g])):
                bysub = []
                for run in range(1, 10):
                    if run == 6: continue
                    
                    TRs = sub_run_TRs[sub][run]
                    tc = bold[sub][run][roi][:, hrf:TRs+hrf]
                    
                    # Filter out invalid voxels
                    valid_mask = np.all(np.isfinite(tc), axis=1)
                    tc_clean = tc[valid_mask, :]
                    
                    if tc_clean.shape[0] < 2:
                        bysub.append(np.full(tc.shape[1] - 2 * window, np.nan))
                    else:
                        # 2. Vectorize Math
                        X = tc_clean[:, :tc_clean.shape[1] - 2 * window]
                        Y = tc_clean[:, window : tc_clean.shape[1] - window]
                        
                        X_c = X - np.mean(X, axis=0)
                        Y_c = Y - np.mean(Y, axis=0)
                        
                        num = np.sum(X_c * Y_c, axis=0)
                        den = np.sqrt(np.sum(X_c**2, axis=0) * np.sum(Y_c**2, axis=0))
                        
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            corr = num / den
                            
                        shifts = 1 - corr
                        bysub.append(shifts)
                        
                diss[g,roi,sub] = bysub
        
        # 3. CRITICAL MEMORY RELEASE
        # Force Python to delete the massive BOLD dictionary and clear RAM 
        # before attempting to load the next group's .pkl file.
        print(f"Group {g} complete. Forcing memory garbage collection...")
        del bold
        gc.collect() 
                
    np.save(os.path.join(OUT_DIR,"1TR_nearbytp_hrf4.npy"), diss, allow_pickle=True)
    print("saved", os.path.join(OUT_DIR,"1TR_nearbytp_hrf4.npy"))
else:
    print("Place Jin's ROIsum_combined_mask_g{1,2,3}.pkl in", LOADED_BOLD_DIR, "then re-run.")

Loading Group 1 into memory (this may take a minute)...
Group 1 loaded. Processing ROIs and Subjects...
Group 1 complete. Forcing memory garbage collection...
Loading Group 2 into memory (this may take a minute)...
Group 2 loaded. Processing ROIs and Subjects...
Group 2 complete. Forcing memory garbage collection...
Loading Group 3 into memory (this may take a minute)...
Group 3 loaded. Processing ROIs and Subjects...
Group 3 complete. Forcing memory garbage collection...
saved /Users/rheamadhogarhia/Desktop/CABLAB RESEARCH/independent ra work/part_4_summer_char_profiles/characterprofilesynchronygit/data/socialaha/data/brain/pattern_shift/1TR_nearbytp_hrf4.npy


In [5]:
# 6.3b · Pattern-shift ~ sentiment-update at hrf=4 — twin of 6.2 (faithful step08 port)
# ---------------------------------------------------------------------------
# Fixes vs. the earlier draft of this cell:
#   (1) NULL: reconstructs the non-aha per-scene null. `scene_null.py` and its output
#       `null_nonearaha_1TR_nb_perscene.npy` are not shipped in Jin's public repo (only the precomputed shuffled array is missing).
#       The per-scene non-aha null is DOCUMENTED: step08's compute_and_save_scene_null reimplements
#       scene_null.py, and the paper Methods (p.23) specify repeating the analysis on non-aha moments
#       (1000 iters). We regenerate it faithfully: draw random NON-aha timepoints (outside any
#       [-5,+3] aha window), take their [-5,+3] windows. Here each
#       scene draws a number of non-aha windows equal to that scene's real aha count and
#       averages them, so the null is structurally identical to the actual statistic and
#       only the aha *positions* are randomized. (Verified line-by-line vs step08 source + paper Methods p.23.)
#   (2) ESTIMATOR: actual & null use ONE tie-aware, NaN-exact Spearman == scipy.spearmanr
#       (verified to 1e-17). Replaces batch_spearman's argsort-argsort ranks.
#   (3) SIGNIFICANCE: Jin's exact method -- per-TR two-tailed permutation p, FDR over the
#       100 CORTICAL ROIs, then the DOUBLE THRESHOLD (intersect step07 elevated-shift ROIs).
#
# RUNTIME: the faithful null is cluster-scale (Jin precomputed it). Long at N_PERM=1000 on a
# laptop; drop N_PERM for a quick look (p-resolution = 1/(N+1)).
import numpy as np, pandas as pd, warnings, os
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore", category=RuntimeWarning)

N_PERM = 1000     # Jin's step08 uses 1000
PATTERN_SHIFT_PATH_HRF4 = os.path.join(JIN_REPO, "data/brain/pattern_shift/1TR_nearbytp_hrf4.npy")
HAVE_SHIFT4 = os.path.exists(PATTERN_SHIFT_PATH_HRF4)
SEED   = 42

if HAVE_DELTAS and HAVE_SHIFT4 and HAVE_AHA and HAVE_EVENT:
    event_idxs = pd.read_csv(EVENT_PATH)
    CHAR_NAMES = ["jack", "kate", "randall", "kevin"]
    def get_char_ids(g, r):      return sorted(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())
    def get_argsort_order(g, r): return np.argsort(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())

    # ---- 1. behavioral sentiment-updates -> (33 subj x 40 scene) grid (matches step08) ----
    all_subs_beh = []
    for g in range(1, 4):
        for sub_str in _JIN_FLIST[g]:
            row = []
            for run in range(2, 11):
                if run == 7: continue
                dissims = []
                for ch in CHAR_NAMES:
                    m = (sent_update['Participant']==sub_str)&(sent_update['Run_to']==run)&(sent_update['Character']==ch)
                    v = sent_update[m]['sent_update'].values
                    dissims.append(v[0] if len(v) > 0 else np.nan)
                for idx in get_char_ids(g, run):
                    row.append(dissims[int(idx)-1] if idx < 5 else (dissims[1]+dissims[3])/2)  # 5 = kate+kevin merge
            all_subs_beh.append(row)
    all_subs_beh = np.array(all_subs_beh)          # (33, 40)

    # ---- 2. character-consensus aha lookups ----
    pattern_shift = np.load(PATTERN_SHIFT_PATH_HRF4, allow_pickle=True).item()
    df_full = pd.read_excel(AHA_ANNOT_PATH)
    if 'character' in df_full.columns:
        df_char = df_full[df_full['character'] == 1].copy()
    else:
        df_full['character_all'] = df_full[['character_rater1','character_rater2','character_rater3']].sum(axis=1)
        df_char = df_full[df_full['character_all'] >= 2].copy()
    df_lookup, null_lookup = {}, {}
    for g in range(1, 4):
        for si, ss in enumerate(_JIN_FLIST[g]):
            ds = df_char[df_char['subject'] == int(ss[4:])]
            for run in range(8):
                ri = run + 2 if run < 5 else run + 3
                dr = ds[ds['run'] == ri]
                null_lookup[(g, si, run)] = dr['TR (run)'].tolist()
                for s in range(1, 6):
                    df_lookup[(g, si, run, s)] = dr[dr['scene'] == s]['TR (run)'].tolist()

    def conv_r2z(r):
        with np.errstate(invalid='ignore', divide='ignore'):
            return 0.5 * (np.log(1 + r) - np.log(1 - r))

    def _win(this_run, trs):
        if len(trs) == 0: return np.full(9, np.nan)
        w = []
        for tp in trs:
            if tp - 5 < 0:               w.append(np.concatenate([np.full(5-tp, np.nan), this_run[0:tp+4]]))
            elif tp + 4 > len(this_run): w.append(np.concatenate([this_run[tp-5:], np.full(4+tp-len(this_run), np.nan)]))
            else:                        w.append(this_run[tp-5:tp+4])
        return np.nanmean(np.array(w), axis=0)

    def _sp_batch(Z, y):
        """Tie-aware, NaN-exact Spearman == scipy.spearmanr (verified 1e-17).
        Clean rows vectorized; rows with extra NaNs fall back to scipy (exact)."""
        out = np.full(Z.shape[0], np.nan); vy = ~np.isnan(y)
        if vy.sum() < 2: return out
        Zc, yc = Z[:, vy], y[vy]
        dirty = np.isnan(Zc).any(axis=1); clean = ~dirty
        if clean.any():
            yr = rankdata(yc); yr = yr - yr.mean()
            Zr = rankdata(Zc[clean], axis=1); Zr = Zr - Zr.mean(axis=1, keepdims=True)
            den = np.linalg.norm(Zr, axis=1) * np.linalg.norm(yr)
            with np.errstate(invalid='ignore', divide='ignore'):
                out[clean] = np.where(den > 1e-8, Zr @ yr / den, np.nan)
        for i in np.where(dirty)[0]:
            m = ~np.isnan(Zc[i])
            if m.sum() >= 2: out[i] = spearmanr(Zc[i][m], yc[m])[0]
        return out

    # ---- 3. ACTUAL per-scene shifts + r-values (== step08 compute_shifts_perscene/compute_rvals) ----
    print("Computing actual neural pattern shifts + r-values...")
    shifts_perscene = np.zeros((116, 33, 40, 9))
    for roi in range(116):
        flat = 0
        for g in range(1, 4):
            for si in range(len(_JIN_FLIST[g])):
                runs = []
                for run in range(8):
                    ri = run + 2 if run < 5 else run + 3
                    tr_ = pattern_shift[g, roi+1, si][run]        # pattern_shift ROI keys are 1..116
                    scenes = np.array([_win(tr_, df_lookup[(g, si, run, s)]) for s in range(1, 6)])
                    runs.append(scenes[get_argsort_order(g, ri)])
                shifts_perscene[roi, flat] = np.vstack(runs); flat += 1
    actual_rvals = np.full((116, 9), np.nan)
    for roi in range(116):
        for tr in range(9):
            z = conv_r2z(shifts_perscene[roi, :, :, tr])          # (33, 40)
            actual_rvals[roi, tr] = np.nanmean([_sp_batch(z[:, sc][None, :], all_subs_beh[:, sc])[0] for sc in range(40)])

    # ---- 4. RECONSTRUCTED non-aha per-scene null (n=N_PERM) ----
    print(f"Reconstructing non-aha null (N_PERM={N_PERM}) -- slow, cluster-scale step...")
    rng = np.random.default_rng(SEED)
    null_rvals = np.zeros((N_PERM, 116, 9))
    for roi in range(116):
        subs = []
        for g in range(1, 4):
            for si in range(len(_JIN_FLIST[g])):
                runs = []
                for run in range(8):
                    ri = run + 2 if run < 5 else run + 3
                    this_run = pattern_shift[g, roi+1, si][run]; L = len(this_run)
                    excl = set()
                    for t in null_lookup[(g, si, run)]:
                        excl.update(range(max(0, t-5), min(L, t+4)))       # exclude [-5,+3] around each char aha
                    avail = np.array([t for t in range(5, L-3) if t not in excl])
                    scenes = np.full((N_PERM, 5, 9), np.nan)
                    for s in range(1, 6):
                        k = len(df_lookup[(g, si, run, s)])                 # real aha count in this scene
                        if k > 0 and len(avail) > 0:
                            ch = rng.choice(avail, size=(N_PERM, k))        # k non-aha draws / iter
                            scenes[:, s-1, :] = np.nanmean(this_run[ch[:, :, None] + np.arange(-5, 4)[None, None, :]], axis=1)
                    runs.append(scenes[:, get_argsort_order(g, ri), :])
                subs.append(np.concatenate(runs, axis=1))                   # (N_PERM, 40, 9)
        null_brain = np.stack(subs, axis=1)                                 # (N_PERM, 33, 40, 9)
        zc = conv_r2z(null_brain)
        for tr in range(9):
            null_rvals[:, roi, tr] = np.nanmean(
                np.stack([_sp_batch(zc[:, :, sc, tr], all_subs_beh[:, sc]) for sc in range(40)], axis=1), axis=1)
        if roi % 20 == 0: print(f"  null roi {roi}/116")

    # ---- 5. SIGNIFICANCE -- Jin's exact method (per-TR FDR over 100 cortical + double threshold) ----
    def _fdr(p):  _, c, _, _ = multipletests(p, alpha=0.05, method='fdr_bh'); return c
    def _twop(real, null): return np.sum(np.abs(null) >= np.abs(real)) / (1 + len(null))
    NROI_COR = 100

    sig_step08 = []
    for tr in range(9):
        pv = [_twop(actual_rvals[roi, tr], null_rvals[:, roi, tr]) for roi in range(NROI_COR)]
        sig_step08.append(np.where(_fdr(pv) < 0.05)[0])

    # step07 elevated-shift ROIs: HARDCODED verbatim from Jin's public step08 (sig_shifts).
    # Purely neural (same 1TR_nearbytp.npy + character-aha annotations) -> valid to reuse here.
    sig_step07 = [
        np.array([], dtype=int),
        np.array([81, 88]),
        np.array([], dtype=int),
        np.array([6,15,20,25,27,29,30,31,32,33,36,38,40,41,42,43,44,46,48,49,66,74,75,76,79,80,81,83,84,85,86,87,88,89,92,93,94,98]),
        np.array([2,6,13,15,20,25,27,29,30,31,33,35,36,39,42,43,44,46,49,65,66,74,75,79,80,81,82,83,84,85,87,88,91,92,98,99]),
        np.array([2,6,13,15,19,20,23,25,28,29,30,31,32,33,35,36,38,40,42,43,44,49,50,51,65,66,74,75,79,80,81,82,83,84,85,87,88,92,94,95,97,98,99]),
        np.array([2,6,13,14,15,25,30,32,35,44,49,65,66,74,79,80,83,84,85,87]),
        np.array([], dtype=int),
        np.array([], dtype=int),
    ]

    sig_double = []
    for tr in range(9):
        s7 = sig_step07[tr]
        if len(s7) == 0: sig_double.append(np.array([], dtype=int)); continue
        pv = np.array([_twop(actual_rvals[roi, tr], null_rvals[:, roi, tr]) for roi in range(NROI_COR)])
        keep = np.where(_fdr(pv[s7]) < 0.05)[0]
        sig_double.append(np.array(s7)[keep])

    print("\nSTEP08 only (per-TR FDR over 100 cortical, two-tailed):")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_step08[tr]):2d}  {list(sig_step08[tr])}")
    print("\nDOUBLE THRESHOLD (step07 intersect step08) -- hrf=4 sensitivity run:")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_double[tr]):2d}  {list(sig_double[tr])}")

    os.makedirs("results/step6", exist_ok=True)
    np.save("results/step6/06__pattern_shift_sentiment_hrf4.npy",
            {"actual_rvals": actual_rvals, "HRF": 4,
             "sig_step08": np.array(sig_step08, dtype=object),
             "sig_double": np.array(sig_double, dtype=object),
             "N_PERM": N_PERM, "SEED": SEED}, allow_pickle=True)
    print("\nSaved -> results/step6/06__pattern_shift_sentiment_hrf4.npy")
else:
    print("Cannot run hrf=4: generate 1TR_nearbytp_hrf4.npy in 6.3a first (needs the BOLD pkls), and run 6.1.")


Computing actual neural pattern shifts + r-values...
Reconstructing non-aha null (N_PERM=1000) -- slow, cluster-scale step...
  null roi 0/116
  null roi 20/116
  null roi 40/116
  null roi 60/116
  null roi 80/116
  null roi 100/116

STEP08 only (per-TR FDR over 100 cortical, two-tailed):
  TR -5:  0  []
  TR -4:  1  [np.int64(19)]
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

DOUBLE THRESHOLD (step07 intersect step08) -- hrf=4 sensitivity run:
  TR -5:  0  []
  TR -4:  0  []
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

Saved -> results/step6/06__pattern_shift_sentiment_hrf4.npy


## 6.4 · Your own `step07` localizer + double threshold (your ROIs, not Jin's hardcoded list)

Jin's double threshold gates `step08` on **his** hardcoded `step07` ROIs (elevated aha-reconfiguration on his
33-subject cohort). For a sentiment analysis on your 29, this recomputes that localizer on **your** data — same
per-scene aha-count-matched null as your `step08`, two-tailed p, FDR over the 100 cortical ROIs per TR — then
gates with it.

**Findings (ran 2026-07-16, N_PERM=10000):**
- Your localizer **replicates Jin's neural map (Fig 4d)**: hrf=3 recovers his [81,88] at TR−4 and shares 36/38 (TR−2), 35/36 (TR−1), 31/43 (TR0) of his ROIs. It's *broader* than his hardcoded list because the per-scene aha-matched null is lower-variance (more powerful) than his per-run single-TR null.
- hrf=4 is exactly a 1-TR shift of hrf=3 (sanity check: hrf4 TR−4 set == hrf3 TR−3 set).
- **Double threshold with YOUR gate is still 0 ROIs at every TR** (hrf=3: `step08` empty; hrf=4: `step08`=[19]@TR−4, and 19 ∉ your TR−4 localizer). So the sentiment→pattern-shift null holds regardless of whose localizer gates it — it wasn't an artifact of importing Jin's foreign ROIs. The **neural** localizer replicates; the **sentiment** coupling does not.

Toggle `HRF` and rerun for each; saved to `results/step6/06__own_step07_{hrf3,hrf4}.npy`.

In [ ]:
# 6.4 · YOUR OWN step07 localizer (29-subject cohort) + double threshold with it, not Jin's hardcoded ROIs.
# Port of Jin's step07 elevated-shift test on YOUR data: actual aha-window shift vs a per-scene non-aha null
# (aha-count-matched, 10k iters -- same null philosophy as your 6.2 step08), two-tailed p, FDR over the 100
# CORTICAL ROIs per TR. Then double-threshold: step08 (6.2/6.3b) INTERSECT your-own-step07.
import numpy as np, pandas as pd, os
from statsmodels.stats.multitest import multipletests
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH

HRF   = "hrf4"          # "hrf4" (Jin's stated lag) or "hrf3" (his step06 code). Run once each for both.
N_PERM, SEED = 10000, 42
PS     = os.path.join(JIN_REPO, "data/brain/pattern_shift", "1TR_nearbytp_hrf4.npy" if HRF=="hrf4" else "1TR_nearbytp.npy")
STEP08 = f"results/step6/06__pattern_shift_sentiment_{HRF}.npy"     # your step08 sig ROIs from 6.2 / 6.3b
_FL = _JIN_FLIST
event_idxs = pd.read_csv(EVENT_PATH)
def _argsort(g,r): return np.argsort(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())

ps = np.load(PS, allow_pickle=True).item()
_df = pd.read_excel(AHA_ANNOT_PATH)
if 'character' in _df.columns: _dc = _df[_df['character']==1].copy()
else:
    _df['character_all']=_df[['character_rater1','character_rater2','character_rater3']].sum(axis=1); _dc=_df[_df['character_all']>=2].copy()
dl, nl = {}, {}
for g in range(1,4):
    for si,ss in enumerate(_FL[g]):
        ds=_dc[_dc['subject']==int(ss[4:])]
        for run in range(8):
            ri=run+2 if run<5 else run+3; dr=ds[ds['run']==ri]; nl[(g,si,run)]=dr['TR (run)'].tolist()
            for s in range(1,6): dl[(g,si,run,s)]=dr[dr['scene']==s]['TR (run)'].tolist()
def _w(tr,trs):
    if len(trs)==0: return np.full(9,np.nan)
    o=[]
    for tp in trs:
        if tp-5<0: o.append(np.concatenate([np.full(5-tp,np.nan),tr[0:tp+4]]))
        elif tp+4>len(tr): o.append(np.concatenate([tr[tp-5:],np.full(4+tp-len(tr),np.nan)]))
        else: o.append(tr[tp-5:tp+4])
    return np.nanmean(np.array(o),axis=0)

# ACTUAL elevated-shift magnitude per ROI x TR = mean aha-window shift over (subject, scene)
shifts=np.full((116,33,40,9),np.nan)
for roi in range(116):
    f=0
    for g in range(1,4):
        for si in range(len(_FL[g])):
            runs=[]
            for run in range(8):
                ri=run+2 if run<5 else run+3; t=ps[g,roi+1,si][run]
                sc=np.array([_w(t,dl[(g,si,run,s)]) for s in range(1,6)]); runs.append(sc[_argsort(g,ri)])
            shifts[roi,f]=np.vstack(runs); f+=1
actual=np.nanmean(shifts,axis=(1,2))                       # (116, 9)

# NULL: per (subject,run,scene) draw k=aha-count non-aha windows, average; mean over (subject,scene)
rng=np.random.default_rng(SEED); nsum=np.zeros((116,N_PERM,9)); ncnt=np.zeros((116,N_PERM,9))
for roi in range(116):
    for g in range(1,4):
        for si in range(len(_FL[g])):
            for run in range(8):
                t=ps[g,roi+1,si][run]; L=len(t); ex=set()
                for a in nl[(g,si,run)]: ex.update(range(max(0,a-5),min(L,a+4)))
                av=np.array([x for x in range(5,L-3) if x not in ex])
                for s in range(1,6):
                    k=len(dl[(g,si,run,s)])
                    if k>0 and len(av)>0:
                        ch=rng.choice(av,size=(N_PERM,k)); wv=np.nanmean(t[ch[:,:,None]+np.arange(-5,4)[None,None,:]],axis=1)
                        m=~np.isnan(wv); nsum[roi]+=np.where(m,wv,0.0); ncnt[roi]+=m
    if roi%20==0: print(f"  {HRF} null roi {roi}/116")
nullmag=np.where(ncnt>0,nsum/np.maximum(ncnt,1),np.nan)

# SIGNIFICANCE: two-tailed p, FDR over 100 cortical per TR
own_step07=[]
for tr in range(9):
    p=np.array([np.sum(np.abs(nullmag[roi,:,tr])>=np.abs(actual[roi,tr]))/(1+N_PERM) for roi in range(100)])
    _,pc,_,_=multipletests(p,alpha=0.05,method="fdr_bh"); own_step07.append(np.where(pc<0.05)[0])
os.makedirs("results/step6",exist_ok=True)
np.save(f"results/step6/06__own_step07_{HRF}.npy",{"sig_step07":np.array(own_step07,dtype=object),"actual":actual,"N_PERM":N_PERM},allow_pickle=True)

print(f"\n== YOUR OWN step07 localizer ({HRF}, N_PERM={N_PERM}); FDR over 100 cortical per TR ==")
for tr in range(9): print(f"  TR {tr-5:+d}: {len(own_step07[tr]):2d}  {[int(x) for x in own_step07[tr]]}")

# DOUBLE THRESHOLD with YOUR localizer: step08 (6.2/6.3b) INTERSECT your-own-step07
s08=np.load(STEP08,allow_pickle=True).item()["sig_step08"]
print(f"\n== DOUBLE THRESHOLD -- step08 INTERSECT your-own-step07 ({HRF}) ==")
for tr in range(9):
    dbl=sorted(set(int(x) for x in s08[tr]) & set(int(x) for x in own_step07[tr]))
    print(f"  TR {tr-5:+d}: step08={sorted(int(x) for x in s08[tr])}  ->  {dbl}")

  hrf4 null roi 0/116
  hrf4 null roi 20/116
  hrf4 null roi 40/116
  hrf4 null roi 60/116
  hrf4 null roi 80/116
  hrf4 null roi 100/116

== YOUR OWN step07 localizer (hrf4, N_PERM=10000); FDR over 100 cortical per TR ==
  TR -5:  2  [74, 88]
  TR -4: 33  [2, 15, 18, 24, 29, 30, 31, 34, 35, 37, 38, 40, 41, 42, 43, 44, 46, 58, 66, 69, 74, 75, 80, 83, 84, 86, 87, 88, 89, 90, 92, 93, 94]
  TR -3: 57  [0, 2, 6, 8, 15, 18, 19, 20, 24, 25, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 50, 51, 65, 66, 68, 72, 74, 75, 76, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 97, 98, 99]
  TR -2: 61  [0, 2, 5, 6, 13, 19, 20, 23, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 50, 51, 54, 65, 66, 68, 70, 72, 74, 75, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 94, 95, 96, 97, 98, 99]
  TR -1: 52  [2, 13, 14, 15, 19, 20, 23, 25, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47

## 6.5 · Perception-change (embedding run-to-run distance) → pattern shift  *(moved from exploratory 09)*

`perception_update = 1 − cosine(emb[run], emb[run−1])` — the RoBERTa-embedding analog of Jin's own impression update (he used USE embeddings). Same `step08` machinery as 6.2, swapping the behavioral side. **6.5a** = actual; **6.5b** = the faithful null + double threshold (RUN LOCALLY, ~hours, like 6.2). Needs `results/embeddings/03b__roberta_embeddings.pkl`.

### 6.5a · Actual correlation

In [1]:
# = Jin step03 cosine_similarity, imported VERBATIM (jin_code/jin_step03.py).
#   Verified identical to the previous local cos() on 5000 vectors (max|diff|=0) and on the
#   zero-norm / NaN edge cases. His version additionally returns NaN for a missing embedding
#   where the local one raised ValueError.
import sys; sys.path.insert(0, 'jin_code')
from jin_step03 import cosine_similarity
import pickle, numpy as np, pandas as pd
from scipy.stats import spearmanr
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH
from roi_labels import ROI_NAME
CHARS=["jack","kate","randall","kevin"]
_FL={1:["sub-1001","sub-1005","sub-1008","sub-1011","sub-1014","sub-1017","sub-1020","sub-1023","sub-1026","sub-1029","sub-1033","sub-1039"],
     2:["sub-2006","sub-2009","sub-2012","sub-2015","sub-2018","sub-2021","sub-2024","sub-2027","sub-2034","sub-2038","sub-2040"],
     3:["sub-3004","sub-3007","sub-3013","sub-3016","sub-3019","sub-3022","sub-3025","sub-3031","sub-3037","sub-3041"]}
emb=pickle.load(open("results/embeddings/03b__roberta_embeddings.pkl","rb"))
ev=pd.read_csv(EVENT_PATH); args=lambda g,r: np.argsort(ev[ev['run']==r][f'g{g}.char'].tolist())
allb=[]                                             # (33,40) perception-update, 40-scene grid
for g in range(1,4):
    for ss in _FL[g]:
        row=[]
        for run in range(2,11):
            if run==7: continue
            dd=[(1-cosine_similarity(_a,_b) if (_a is not None and _b is not None) else np.nan)
            for ch in CHARS for _a,_b in [(emb.get((ss,run,ch)),emb.get((ss,run-1,ch)))]]
            for idx in sorted(ev[ev['run']==run][f'g{g}.char'].tolist()):
                row.append(dd[int(idx)-1] if idx<5 else (dd[1]+dd[3])/2)
        allb.append(row)
allb=np.array(allb)
ps=np.load(JIN_REPO+"/data/brain/pattern_shift/1TR_nearbytp_hrf4.npy",allow_pickle=True).item()
df=pd.read_excel(AHA_ANNOT_PATH)
if 'character' in df.columns: dc=df[df['character']==1].copy()
else: df['ca']=df[['character_rater1','character_rater2','character_rater3']].sum(1); dc=df[df['ca']>=2].copy()
dl={}
for g in range(1,4):
    for si,ss in enumerate(_FL[g]):
        ds=dc[dc['subject']==int(ss[4:])]
        for run in range(8):
            ri=run+2 if run<5 else run+3; drr=ds[ds['run']==ri]
            for s in range(1,6): dl[(g,si,run,s)]=drr[drr['scene']==s]['TR (run)'].tolist()
def win(tr,trs):
    if len(trs)==0: return np.full(9,np.nan)
    w=[]
    for tp in trs:
        if tp-5<0: w.append(np.concatenate([np.full(5-tp,np.nan),tr[0:tp+4]]))
        elif tp+4>len(tr): w.append(np.concatenate([tr[tp-5:],np.full(4+tp-len(tr),np.nan)]))
        else: w.append(tr[tp-5:tp+4])
    return np.nanmean(np.array(w),0)
r2z=lambda r: 0.5*(np.log(1+r)-np.log(1-r))
def nsp(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float); i=np.union1d(np.where(np.isnan(a))[0],np.where(np.isnan(b))[0])
    a,b=np.delete(a,i),np.delete(b,i); return spearmanr(a,b)[0] if len(a)>2 else np.nan
sh=np.full((116,33,40,9),np.nan)
for roi in range(116):
    f=0
    for g in range(1,4):
        for si in range(len(_FL[g])):
            runs=[]
            for run in range(8):
                ri=run+2 if run<5 else run+3; t=ps[g,roi+1,si][run]
                runs.append(np.array([win(t,dl[(g,si,run,s)]) for s in range(1,6)])[args(g,ri)])
            sh[roi,f]=np.vstack(runs); f+=1
with np.errstate(all="ignore"):
    act=np.array([[np.nanmean([nsp(r2z(sh[roi,:,:,tr])[:,sc],allb[:,sc]) for sc in range(40)]) for tr in range(9)] for roi in range(116)])
np.save("results/step6/06__perceptionchange_actual_hrf4.npy",{"actual":act,"allb":allb},allow_pickle=True)
pk=np.unravel_index(np.argmax(np.abs(np.nan_to_num(act))),act.shape)
print(f"ACTUAL pattern-shift ~ perception-change: max|r|={act[pk]:+.3f} at ROI {pk[0]} ({ROI_NAME.get(pk[0])}), TR {pk[1]-5:+d}")
for tr in range(9):
    top=np.argsort(-np.abs(np.nan_to_num(act[:,tr])))[:3]
    print(f"  TR {tr-5:+d}: "+", ".join(f"{i}({act[i,tr]:+.2f})" for i in top))

/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_10879/352412729.py:47: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.array(w),0)


ACTUAL pattern-shift ~ perception-change: max|r|=+0.366 at ROI 110 (LH post.Thalamus), TR +2
  TR -5: 77(+0.25), 94(-0.20), 67(+0.20)
  TR -4: 28(+0.28), 17(+0.25), 56(+0.22)
  TR -3: 63(+0.21), 94(+0.21), 44(-0.21)
  TR -2: 112(+0.27), 12(+0.24), 28(-0.22)
  TR -1: 36(+0.24), 23(+0.24), 17(+0.20)
  TR +0: 22(+0.22), 10(+0.22), 20(+0.21)
  TR +1: 28(+0.25), 95(+0.22), 109(+0.21)
  TR +2: 110(+0.37), 106(+0.22), 101(+0.21)
  TR +3: 72(+0.27), 82(+0.22), 76(+0.18)


### 6.5b · Significance — faithful null + double threshold  ⚠ RUN LOCALLY

In [2]:
# 9.2b · perception-change significance -- run after 9.2a (uses allb, sh). N_PERM=1000, slow (local).
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
N_PERM,SEED=1000,42
_FL={1:list(range(12)),2:list(range(11)),3:list(range(10))}   # sizes only
# rebuild aha-window null pool from the same lookups as 9.2a
from config import JIN_REPO, AHA_ANNOT_PATH
import pandas as pd
FL={1:["sub-1001","sub-1005","sub-1008","sub-1011","sub-1014","sub-1017","sub-1020","sub-1023","sub-1026","sub-1029","sub-1033","sub-1039"],
    2:["sub-2006","sub-2009","sub-2012","sub-2015","sub-2018","sub-2021","sub-2024","sub-2027","sub-2034","sub-2038","sub-2040"],
    3:["sub-3004","sub-3007","sub-3013","sub-3016","sub-3019","sub-3022","sub-3025","sub-3031","sub-3037","sub-3041"]}
ps=np.load(JIN_REPO+"/data/brain/pattern_shift/1TR_nearbytp_hrf4.npy",allow_pickle=True).item()
df=pd.read_excel(AHA_ANNOT_PATH)
if 'character' in df.columns: dc=df[df['character']==1].copy()
else: df['ca']=df[['character_rater1','character_rater2','character_rater3']].sum(1); dc=df[df['ca']>=2].copy()
import pandas as pd
from config import EVENT_PATH
ev=pd.read_csv(EVENT_PATH); args=lambda g,r: np.argsort(ev[ev['run']==r][f'g{g}.char'].tolist())
nl={}; dl={}
for g in range(1,4):
    for si,ss in enumerate(FL[g]):
        ds=dc[dc['subject']==int(ss[4:])]
        for run in range(8):
            ri=run+2 if run<5 else run+3; drr=ds[ds['run']==ri]; nl[(g,si,run)]=drr['TR (run)'].tolist()
            for s in range(1,6): dl[(g,si,run,s)]=drr[drr['scene']==s]['TR (run)'].tolist()
r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))
def sp(Z,y):
    out=np.full(Z.shape[0],np.nan); vy=~np.isnan(y)
    if vy.sum()<2: return out
    Zc,yc=Z[:,vy],y[vy]; clean=~np.isnan(Zc).any(1)
    if clean.any():
        yr=rankdata(yc)-np.mean(rankdata(yc)); Zr=rankdata(Zc[clean],axis=1); Zr=Zr-Zr.mean(1,keepdims=True)
        den=np.linalg.norm(Zr,axis=1)*np.linalg.norm(yr)
        with np.errstate(all="ignore"): out[np.where(clean)[0]]=np.where(den>1e-8,Zr@yr/den,np.nan)
    for i in np.where(~clean)[0]:
        m=~np.isnan(Zc[i]); out[i]=spearmanr(Zc[i][m],yc[m])[0] if m.sum()>2 else np.nan
    return out
rng=np.random.default_rng(SEED); null=np.zeros((N_PERM,116,9))
for roi in range(116):
    subs=[]
    for g in range(1,4):
        for si in range(len(FL[g])):
            runs=[]
            for run in range(8):
                ri=run+2 if run<5 else run+3; this=ps[g,roi+1,si][run]; L=len(this); ex=set()
                for t in nl[(g,si,run)]: ex.update(range(max(0,t-5),min(L,t+4)))
                av=np.array([t for t in range(5,L-3) if t not in ex]); scenes=np.full((N_PERM,5,9),np.nan)
                for s in range(1,6):
                    k=len(dl[(g,si,run,s)])
                    if k>0 and len(av)>0:
                        ch=rng.choice(av,size=(N_PERM,k)); scenes[:,s-1,:]=np.nanmean(this[ch[:,:,None]+np.arange(-5,4)[None,None,:]],1)
                runs.append(scenes[:,args(g,ri),:])
            subs.append(np.concatenate(runs,axis=1))
    nb=np.stack(subs,1); zc=r2z(nb)
    for tr in range(9):
        null[:,roi,tr]=np.nanmean(np.stack([sp(zc[:,:,sc,tr],allb[:,sc]) for sc in range(40)],1),1)
    if roi%20==0: print("null roi",roi)
def fdr(p): _,c,_,_=multipletests(p,alpha=0.05,method="fdr_bh"); return c
twop=lambda real,nu: np.sum(np.abs(nu)>=np.abs(real))/(1+len(nu))
sig08=[np.where(fdr([twop(act[roi,tr],null[:,roi,tr]) for roi in range(100)])<0.05)[0] for tr in range(9)]
s7=np.load("results/step6/06__own_step07_hrf4.npy",allow_pickle=True).item()["sig_step07"]
sig_dbl=[np.array(sorted(set(int(x) for x in sig08[tr])&set(int(x) for x in s7[tr]))) for tr in range(9)]
print("\nperception-change step08-only (per-TR FDR/100 cortical):")
for tr in range(9): print(f"  TR {tr-5:+d}: {[int(x) for x in sig08[tr]]}")
print("DOUBLE THRESHOLD (∩ your step07):")
for tr in range(9): print(f"  TR {tr-5:+d}: {[int(x) for x in sig_dbl[tr]]}")
np.save("results/step6/06__perceptionchange_sig_hrf4.npy",{"sig_step08":np.array(sig08,dtype=object),"sig_double":np.array(sig_dbl,dtype=object)},allow_pickle=True)

/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_14838/1029980978.py:52: RuntimeWarning: Mean of empty slice
  ch=rng.choice(av,size=(N_PERM,k)); scenes[:,s-1,:]=np.nanmean(this[ch[:,:,None]+np.arange(-5,4)[None,None,:]],1)
/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_14838/1029980978.py:27: RuntimeWarning: invalid value encountered in log
  r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))


null roi 0


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_14838/1029980978.py:27: RuntimeWarning: divide by zero encountered in log
  r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))


null roi 20
null roi 40
null roi 60
null roi 80
null roi 100

perception-change step08-only (per-TR FDR/100 cortical):
  TR -5: []
  TR -4: []
  TR -3: []
  TR -2: []
  TR -1: []
  TR +0: []
  TR +1: []
  TR +2: []
  TR +3: []
DOUBLE THRESHOLD (∩ your step07):
  TR -5: []
  TR -4: []
  TR -3: []
  TR -2: []
  TR -1: []
  TR +0: []
  TR +1: []
  TR +2: []
  TR +3: []


## 6.6 · POSITIVE CONTROL — can `06` recover Jin's *known* Fig-4 effect?

The IS-RSA side has a positive control (`04a` reproduces Jin's Fig 2). The pattern-shift side has **none** — so the
sentiment (6.2) and perception-change (6.5) nulls can't be read as true absence until we show this pipeline can
recover Jin's **impression-update → pattern-shift** result (his Fig 4). This runs his USE impression-update
(`speech-embeddings_byrun_bychar.npy`) through the identical port + reconstructed null.

> [!warning] Diagnostic (6.6a, ran): the impression-update signal is **not concentrated** in Jin's step07 regions —
> mean|r| there ≈ all-cortical (ratio 0.92–1.13). So Jin's own effect looks diffuse through this pipeline, which
> **strongly suggests the control will fail** (i.e., the reconstructed null / hrf / window extraction can't recover
> Fig 4). If 6.6b confirms 0 double-threshold ROIs, the sentiment/perception nulls are **not interpretable as
> absence** — that becomes a required caveat (or a reason to get Jin's real `scene_null.py`).

### 6.6a · Impression-update actual + concentration diagnostic

In [4]:
# 6.6a · POSITIVE CONTROL (actual) -- Jin's USE impression-update ~ pattern shift, HIS pipeline (hrf=3).
# If 06 cannot recover Jin's known Fig-4 effect, the sentiment/perception nulls are NOT interpretable as absence.
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH
imp=np.load(JIN_REPO+"/data/beh/embeddings/speech-embeddings_byrun_bychar.npy",allow_pickle=True).item()
ev=pd.read_csv(EVENT_PATH); _args=lambda g,r: np.argsort(ev[ev['run']==r][f'g{g}.char'].tolist())
def _cos(a,b):
    a=np.asarray(a,float);b=np.asarray(b,float);d=np.linalg.norm(a)*np.linalg.norm(b);return float(a@b/d) if d else np.nan
allb_imp=[]
for g in range(1,4):
    for si in range(len(_JIN_FLIST[g])):
        row=[]
        for run in range(2,11):
            if run==7: continue
            dd=[1-_cos(imp[g,si,run][ch],imp[g,si,run-1][ch]) for ch in range(4)]
            for idx in sorted(ev[ev['run']==run][f'g{g}.char'].tolist()):
                row.append(dd[int(idx)-1] if idx<5 else (dd[1]+dd[3])/2)
        allb_imp.append(row)
allb_imp=np.array(allb_imp)
ps3=np.load(JIN_REPO+"/data/brain/pattern_shift/1TR_nearbytp.npy",allow_pickle=True).item()   # hrf=3
_df=pd.read_excel(AHA_ANNOT_PATH); _dc=_df[_df['character']==1].copy() if 'character' in _df.columns else None
if _dc is None: _df['ca']=_df[['character_rater1','character_rater2','character_rater3']].sum(1); _dc=_df[_df['ca']>=2].copy()
dl3={}
for g in range(1,4):
    for si,ss in enumerate(_JIN_FLIST[g]):
        ds=_dc[_dc['subject']==int(ss[4:])]
        for run in range(8):
            ri=run+2 if run<5 else run+3; drr=ds[ds['run']==ri]
            for s in range(1,6): dl3[(g,si,run,s)]=drr[drr['scene']==s]['TR (run)'].tolist()
def _win(tr,trs):
    if len(trs)==0: return np.full(9,np.nan)
    w=[]
    for tp in trs:
        if tp-5<0: w.append(np.concatenate([np.full(5-tp,np.nan),tr[0:tp+4]]))
        elif tp+4>len(tr): w.append(np.concatenate([tr[tp-5:],np.full(4+tp-len(tr),np.nan)]))
        else: w.append(tr[tp-5:tp+4])
    return np.nanmean(np.array(w),0)
_r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))
def _nsp(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float); i=np.union1d(np.where(np.isnan(a))[0],np.where(np.isnan(b))[0])
    a,b=np.delete(a,i),np.delete(b,i); return spearmanr(a,b)[0] if len(a)>2 else np.nan
sh_imp=np.full((116,33,40,9),np.nan)
for roi in range(116):
    f=0
    for g in range(1,4):
        for si in range(len(_JIN_FLIST[g])):
            runs=[]
            for run in range(8):
                ri=run+2 if run<5 else run+3; t=ps3[g,roi+1,si][run]
                runs.append(np.array([_win(t,dl3[(g,si,run,s)]) for s in range(1,6)])[_args(g,ri)])
            sh_imp[roi,f]=np.vstack(runs); f+=1
with np.errstate(all="ignore"):
    act_imp=np.array([[np.nanmean([_nsp(_r2z(sh_imp[roi,:,:,tr])[:,sc],allb_imp[:,sc]) for sc in range(40)]) for tr in range(9)] for roi in range(116)])
np.save("results/step6/06__impression_control_actual_hrf3.npy",{"actual":act_imp,"allb":allb_imp},allow_pickle=True)
s7=[np.array([],int),np.array([81,88]),np.array([],int),np.array([6,15,20,25,27,29,30,31,32,33,36,38,40,41,42,43,44,46,48,49,66,74,75,76,79,80,81,83,84,85,86,87,88,89,92,93,94,98]),np.array([2,6,13,15,20,25,27,29,30,31,33,35,36,39,42,43,44,46,49,65,66,74,75,79,80,81,82,83,84,85,87,88,91,92,98,99]),np.array([2,6,13,15,19,20,23,25,28,29,30,31,32,33,35,36,38,40,42,43,44,49,50,51,65,66,74,75,79,80,81,82,83,84,85,87,88,92,94,95,97,98,99]),np.array([2,6,13,14,15,25,30,32,35,44,49,65,66,74,79,80,83,84,85,87]),np.array([],int),np.array([],int)]
print("POSITIVE CONTROL -- Jin USE impression-update ~ pattern shift (hrf=3), ACTUAL:")
print("  is the signal CONCENTRATED in Jin's step07 regions (where Fig 4 lives)?")
for tr in range(9):
    s=[int(x) for x in s7[tr]]
    if not s: print(f"  TR {tr-5:+d}: (no step07 ROIs)"); continue
    inr=np.nanmean([abs(act_imp[r,tr]) for r in s]); allc=np.nanmean(np.abs(np.nan_to_num(act_imp[:100,tr])))
    print(f"  TR {tr-5:+d}: mean|r| step07-ROIs={inr:.3f}  vs all-cortical={allc:.3f}  (ratio {inr/allc:.2f})")
print("  ratio ~1.0 => signal NOT concentrated where Jin found it => control likely fails; run 6.6b to confirm.")

/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_33179/2359932534.py:38: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.array(w),0)


POSITIVE CONTROL -- Jin USE impression-update ~ pattern shift (hrf=3), ACTUAL:
  is the signal CONCENTRATED in Jin's step07 regions (where Fig 4 lives)?
  TR -5: (no step07 ROIs)
  TR -4: mean|r| step07-ROIs=0.036  vs all-cortical=0.079  (ratio 0.45)
  TR -3: (no step07 ROIs)
  TR -2: mean|r| step07-ROIs=0.074  vs all-cortical=0.065  (ratio 1.13)
  TR -1: mean|r| step07-ROIs=0.080  vs all-cortical=0.077  (ratio 1.04)
  TR +0: mean|r| step07-ROIs=0.064  vs all-cortical=0.070  (ratio 0.92)
  TR +1: mean|r| step07-ROIs=0.078  vs all-cortical=0.073  (ratio 1.06)
  TR +2: (no step07 ROIs)
  TR +3: (no step07 ROIs)
  ratio ~1.0 => signal NOT concentrated where Jin found it => control likely fails; run 6.6b to confirm.


### 6.6b · Impression-update significance — the Fig-4 recovery test  ⚠ RUN LOCALLY (~hours)

In [5]:
# 6.6b · POSITIVE CONTROL (significance) -- run after 6.6a (uses allb_imp, sh_imp, s7). N_PERM=1000, LOCAL ~hours.
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH
import pandas as pd
N_PERM,SEED=1000,42
ps3=np.load(JIN_REPO+"/data/brain/pattern_shift/1TR_nearbytp.npy",allow_pickle=True).item()
_df=pd.read_excel(AHA_ANNOT_PATH); _dc=_df[_df['character']==1].copy() if 'character' in _df.columns else None
if _dc is None: _df['ca']=_df[['character_rater1','character_rater2','character_rater3']].sum(1); _dc=_df[_df['ca']>=2].copy()
ev=pd.read_csv(EVENT_PATH); _args=lambda g,r: np.argsort(ev[ev['run']==r][f'g{g}.char'].tolist())
nl={}; dl={}
for g in range(1,4):
    for si,ss in enumerate(_JIN_FLIST[g]):
        ds=_dc[_dc['subject']==int(ss[4:])]
        for run in range(8):
            ri=run+2 if run<5 else run+3; drr=ds[ds['run']==ri]; nl[(g,si,run)]=drr['TR (run)'].tolist()
            for s in range(1,6): dl[(g,si,run,s)]=drr[drr['scene']==s]['TR (run)'].tolist()
_r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))
def _sp(Z,y):
    out=np.full(Z.shape[0],np.nan); vy=~np.isnan(y)
    if vy.sum()<2: return out
    Zc,yc=Z[:,vy],y[vy]; clean=~np.isnan(Zc).any(1)
    if clean.any():
        yr=rankdata(yc)-np.mean(rankdata(yc)); Zr=rankdata(Zc[clean],axis=1); Zr=Zr-Zr.mean(1,keepdims=True)
        den=np.linalg.norm(Zr,axis=1)*np.linalg.norm(yr)
        with np.errstate(all="ignore"): out[np.where(clean)[0]]=np.where(den>1e-8,Zr@yr/den,np.nan)
    for i in np.where(~clean)[0]:
        m=~np.isnan(Zc[i]); out[i]=spearmanr(Zc[i][m],yc[m])[0] if m.sum()>2 else np.nan
    return out
rng=np.random.default_rng(SEED); null=np.zeros((N_PERM,116,9))
for roi in range(116):
    subs=[]
    for g in range(1,4):
        for si in range(len(_JIN_FLIST[g])):
            runs=[]
            for run in range(8):
                ri=run+2 if run<5 else run+3; this=ps3[g,roi+1,si][run]; L=len(this); ex=set()
                for t in nl[(g,si,run)]: ex.update(range(max(0,t-5),min(L,t+4)))
                av=np.array([t for t in range(5,L-3) if t not in ex]); scenes=np.full((N_PERM,5,9),np.nan)
                for s in range(1,6):
                    k=len(dl[(g,si,run,s)])
                    if k>0 and len(av)>0:
                        ch=rng.choice(av,size=(N_PERM,k)); scenes[:,s-1,:]=np.nanmean(this[ch[:,:,None]+np.arange(-5,4)[None,None,:]],1)
                runs.append(scenes[:,_args(g,ri),:])
            subs.append(np.concatenate(runs,axis=1))
    nb=np.stack(subs,1); zc=_r2z(nb)
    for tr in range(9): null[:,roi,tr]=np.nanmean(np.stack([_sp(zc[:,:,sc,tr],allb_imp[:,sc]) for sc in range(40)],1),1)
    if roi%20==0: print("null roi",roi)
def _fdr(p): _,c,_,_=multipletests(p,alpha=0.05,method="fdr_bh"); return c
_twop=lambda real,nu: np.sum(np.abs(nu)>=np.abs(real))/(1+len(nu))
sig08=[np.where(_fdr([_twop(act_imp[roi,tr],null[:,roi,tr]) for roi in range(100)])<0.05)[0] for tr in range(9)]
sig_dbl=[np.array(sorted(set(int(x) for x in sig08[tr])&set(int(x) for x in s7[tr]))) for tr in range(9)]
print("\nIMPRESSION-UPDATE positive control -- step08-only:")
for tr in range(9): print(f"  TR {tr-5:+d}: {[int(x) for x in sig08[tr]]}")
print("DOUBLE THRESHOLD (Jin Fig-4 recovery test):")
for tr in range(9): print(f"  TR {tr-5:+d}: {[int(x) for x in sig_dbl[tr]]}")
print("\n>>> If these are non-empty (esp. TR -2/-1/0), 06 recovers Jin's effect -> sentiment/perception nulls are MEANINGFUL.")
print(">>> If all empty, the reconstructed null is too conservative -> 06 nulls are UNINTERPRETABLE (needs Jin's scene_null).")
np.save("results/step6/06__impression_control_sig_hrf3.npy",{"sig_step08":np.array(sig08,dtype=object),"sig_double":np.array(sig_dbl,dtype=object)},allow_pickle=True)

/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_33179/3837869425.py:44: RuntimeWarning: Mean of empty slice
  ch=rng.choice(av,size=(N_PERM,k)); scenes[:,s-1,:]=np.nanmean(this[ch[:,:,None]+np.arange(-5,4)[None,None,:]],1)
/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_33179/3837869425.py:19: RuntimeWarning: invalid value encountered in log
  _r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))


null roi 0


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_33179/3837869425.py:19: RuntimeWarning: divide by zero encountered in log
  _r2z=lambda r:0.5*(np.log(1+r)-np.log(1-r))


null roi 20
null roi 40
null roi 60
null roi 80
null roi 100

IMPRESSION-UPDATE positive control -- step08-only:
  TR -5: [52, 78]
  TR -4: []
  TR -3: []
  TR -2: []
  TR -1: []
  TR +0: [15]
  TR +1: []
  TR +2: []
  TR +3: []
DOUBLE THRESHOLD (Jin Fig-4 recovery test):
  TR -5: []
  TR -4: []
  TR -3: []
  TR -2: []
  TR -1: []
  TR +0: [15]
  TR +1: []
  TR +2: []
  TR +3: []

>>> If these are non-empty (esp. TR -2/-1/0), 06 recovers Jin's effect -> sentiment/perception nulls are MEANINGFUL.
>>> If all empty, the reconstructed null is too conservative -> 06 nulls are UNINTERPRETABLE (needs Jin's scene_null).
